<!-- # ColorBrowserAgent: An Intelligent GUI Agent for Complex Long-Horizon Web Automation -->
# ColorBrowserAgent：用于复杂长期网络自动化的智能图形界面代理

- [论文](https://arxiv.org/abs/2601.07262)
- [代码](https://github.com/MadeAgents/browser-agent.git)

## 摘要

网页浏览器是人类日常活动的核心交互界面，其自动化技术因此成为以人为本的人工智能领域的重要研究方向。尽管大语言模型已赋能智能体实现网页图形用户界面自主交互，但长程任务不稳定性及网页设计异构性问题，严重制约了其真实场景可靠性。

本文提出**协同自主式网页智能体ColorBrowserAgent**，整合两大核心机制：
1.  **渐进式进度摘要**：模拟人类短时记忆，保障长程交互连贯性；
2.  **人在环知识适配**：仅在必要时引入专家干预，填补复杂环境下的知识缺口。

这种协同设计无需大量重训练即可让智能体学习人类经验，高效结合了AI的规模化优势与人类认知的自适应能力。基于GPT-5在WebArena基准测试的结果显示，该智能体任务成功率达**71.2%**，刷新当前最优纪录，印证了交互式人类辅助对提升网页自动化鲁棒性的显著作用。



## 1 引言

网页浏览器已成为人机交互的通用工具，是信息检索、复杂专业流程等各类活动的核心入口。因此，浏览器任务自动化对提升人类生产力具有巨大潜力。近年来大语言模型（LLM）的兴起加速了这一进程，使智能体能够解析自然语言指令并直接操控图形用户界面（GUI）。然而，从被动工具向完全自主智能体的跨越仍面临巨大挑战，尤其是在网页这一“开放世界”中——网页的设计初衷是服务人类视觉认知，而非机器解析。

当前智能体的效能受两大核心问题制约：
1.  **长程稳定性不足**：复杂任务往往需要多步连续交互，超出大语言模型的上下文窗口上限，导致智能体出现“决策漂移”，偏离初始目标。
2.  **网站异构性壁垒**：网页设计范式与领域专属逻辑高度多样，通用模型难以适配。人类可凭直觉快速适应新界面，而智能体若无特定先验知识则极易失效。

<div style="background-color: white; padding: 10px; margin:auto; width: 80%; text-align: center;">
    <img src="./assets/intro.png" />
    <span style="color: black;"><strong>图1:</strong> 复杂网络自动化中的挑战与拟议的 ColorBrowserAgent 框架。
    </span>
</div>

针对上述问题，本文提出面向**协同自主**的智能体框架**ColorBrowserAgent**。考虑到异构环境下完全自主执行的脆弱性，该框架采用**以人为本**的设计思路，将人类专业知识融入智能体工作流程，核心机制包括：
1.  **渐进式进度摘要**：设计类人类工作记忆的记忆机制，持续生成任务执行的连贯摘要，避免长任务中的上下文溢出问题。
2.  **人在环知识适配**：智能体检测到异常时主动请求人类协助，将专家提供的自然语言“经验提示”存入自适应知识库。该机制无需重新训练即可让系统适配不同网站的特有逻辑，实现“人机协作”而非“人机替代”的模式革新，在保障任务鲁棒性的同时降低用户认知负担。

本文核心贡献如下：
1.  **提出渐进式进度摘要机制**：设计摘要智能体，将交互历史实时压缩为固定长度摘要，有效解决复杂长程任务中的进度混乱与上下文溢出问题，彻底规避决策漂移。
2.  **提出人在环知识适配框架**：结合视觉语言模型（VLM）判别器与规则判别器，精准识别需人工介入的场景，将领域专家知识融入系统。该方法无需大量训练或繁琐人工操作，即可让智能体掌握并应用网站专属先验知识，攻克异构性难题。
3.  **工程优化与SOTA性能验证**：实现动作空间扩展、环境定制化容错等关键工程优化，显著提升系统鲁棒性与效率。在WebArena基准测试集上的严格评估表明，ColorBrowserAgent任务成功率达**71.2%**，超越现有基线模型，刷新当前最优纪录。

## 2 相关工作

### 2.1 基于大语言模型的网页智能体架构
网页自动化技术已从启发式脚本演进为大语言模型（LLM）驱动的自主智能体。早期研究的核心是构建WebArena、Mind2Web等评测基准，界定网页导航任务的范畴。在此基础上，相关研究转向专用智能体架构开发：WebVoyager与AutoWebGLM证实了多模态感知能力对解析复杂HTML与视觉布局的必要性；近期研究则聚焦大动作模型（LAM），例如ActionStudio提出轻量化专用动作令牌训练框架，另有研究探索异构大语言模型的协同机制以平衡推理深度与效率。上述研究表明，网页智能体的技术瓶颈正从简单指令遵循，转向在异构环境中精准执行原子动作。

### 2.2 长程规划与记忆管理
复杂多步网页任务要求智能体维持时序连贯性，并高效管理持续增长的交互历史。针对大语言模型上下文窗口的固有局限，现有研究提出多种状态跟踪机制：WebExplorer采用“探索-演进”策略保障长时程策略稳定性；ReSum通过递归摘要提取关键状态转移信息，避免信息过载；还有研究设计层级化过程记忆以规整长期目标。这些方法均表明，实现可靠的长程任务自动化，关键在于智能体对过往经验的抽象能力，以及对任务进度一致“信念状态”的维持能力——这正是结构化摘要技术需解决的核心问题。

### 2.3 环境适配与反馈机制
网页结构异构性与真实界面的随机性，是网页自动化面临的重要挑战。为提升系统鲁棒性，研究者将外部知识与人类经验融入智能体流程：KG-RAG借助结构化知识图谱辅助跨域推理；人在环（HITL）范式被用于处理智能体当前策略无法应对的边缘场景，例如WebCoach构建基于人类反馈的跨会话学习框架，支持智能体适配新型界面；另有研究证实，稀疏人工修正可显著加速长程任务技能的习得。综上，设计能自主识别执行不确定性并发起定向人工干预的机制，是构建高可靠网页智能体的关键研究方向。

## 3 系统架构

为应对长程网页任务的复杂性，**ColorBrowserAgent**采用**多智能体协同架构**（见[图2]()），整合两大核心方法：用于全局上下文管理的**渐进式进度摘要**，以及用于消除网站异构性知识鸿沟的**人机交互知识增强**。

<div style="background-color: white; padding: 10px; margin:auto; width: 80%; text-align: center;">
    <img src="./assets/framework1.png" />
    <span style="color: black;"><strong>图2:</strong> ColorBrowserAgent框架。该双智能体架构包含两大核心模块：（1）保障长程稳定性的渐进式进度摘要；（2）人机知识适配（HITL-KA）来克服网站异构性挑战。
    </span>
</div>

__问题建模__  
本文将网页自动化任务建模为**部分可观测马尔可夫决策过程（POMDP）**，其元组形式为 $(\mathcal{S}, \mathcal{A}, \mathcal{O}, \mathcal{T}, \mathcal{R}, G)$。其中：
- $\mathcal{S}$ 为环境隐状态空间；
- $\mathcal{A}$ 为动作空间（如点击、输入）；
- $\mathcal{O}$ 为多模态观测空间。

在每个时间步 $t$，智能体接收观测 $o_t = (D_t, V_t)$，$D_t$ 为可访问性树，$V_t$ 为叠加标记集（SoM）标注的截图。智能体决策基于累积交互历史 $h_t = (o_0, a_0, \dots, o_t)$，优化目标是学习最优策略 $\pi^*$ 以最大化任务期望成功率：
$$\pi^* = \operatorname*{arg\,max}_{\pi} \mathbb{E}_{\tau \sim \pi} \left[ \mathcal{R}(\tau) \mid G \right]$$
式中 $\tau$ 为执行轨迹，$\mathcal{R}(\cdot) \in \{0, 1\}$ 为指示任务完成与否的稀疏奖励函数。

__双智能体控制循环__
直接优化公式中的整体策略 $\pi(a_t | h_t, G)$ 在长程任务中难以实现——交互历史长度 $|h_t|$ 随时间线性增长，易引发上下文溢出与推理性能下降。为此，本文提出**分解式双智能体架构**，通过时序抽象与外部知识增强实现全局策略近似，将实时上下文复杂度从 $O(t)$ 降至 $O(1)$。该架构将控制循环解耦为三个耦合函数：
1.  **上下文感知知识检索**
    执行前，从自适应知识库 $\mathcal{K}$ 中检索领域专属约束 $k_t$，引导策略生成有效动作：
    $$k_t = \text{Retrieve}(o_t, \mathcal{K})$$
2.  **渐进式状态压缩（摘要智能体）**
    摘要智能体不直接处理完整历史 $h_t$，而是维护一个紧凑的循环记忆状态 $m_t$，同时承担监督角色，负责更新任务摘要并提供条件性修正指导：
    $$m_t = \pi_{\text{sum}}(m_{t-1}, o_t, a_{t-1}, G)$$
3.  **知识驱动动作执行（操作智能体）**
    操作智能体仅基于局部观测、全局摘要与检索到的经验提示执行精准动作，实现对原策略的近似：
    $$a_t \sim \pi_{\text{op}}(a_t | o_t, m_t, k_t, G) \approx \pi(a_t | h_t, G)$$


## 4 核心方法论

本节详细阐述ColorBrowserAgent的核心技术方案，包括保障长程任务连贯性的**渐进式进度摘要机制**、赋能智能体适配异构网页环境的**人机知识适配**，以及提升系统鲁棒性与执行效率的**工程优化策略**。

### 4.1 渐进式进度摘要
长程网页自动化任务中，由HTML快照、可访问性树和交互日志构成的轨迹历史 $h_t$ 随时间呈 $O(T)$ 线性增长。若直接输入模型，会迅速耗尽上下文窗口并引入噪声，导致智能体出现决策漂移，偏离初始目标或重复无效操作。

为此，本文设计**专用摘要智能体**（$\pi_{\text{sum}}$），与操作智能体（$\pi_{\text{op}}$）交替协同工作。该智能体不保留完整原始历史，而是在每个时间步将轨迹压缩为结构化、可更新的状态表示 $m_t$，其更新函数定义如下：
$$m_t = \pi_{\text{sum}}(m_{t-1}, o_t, a_{t-1}, G)$$

该函数需满足两大核心特性：
1.  **信息保真度**：保留与任务相关的关键信息（如订单号、价格）及子任务相对于全局目标 $G$ 的完成状态；
2.  **长度可控性**：将输出长度限制在近似恒定范围内，使操作智能体的输入复杂度降至 $O(|o_t| + |m_t|)$，而非 $O(T)$。

__结构化摘要模板:__  
为规范记忆表示并降低幻觉风险，本文为 $m_t$ 设计了模式驱动的模板，包含三个核心字段：
1.  **当前进度**：自适应、层级化的任务执行记录。不同于静态日志，该字段根据轨迹长度动态调整粒度——子任务初期记录细粒度操作，执行过程中逐步提炼为高层摘要，确保记忆占用空间恒定。
2.  **当前状态分析**：对当前页面状态 $o_t$ 的策略性评估，明确判断智能体是否处于任务关键路径，提炼与当前子任务相关的核心交互元素，为操作智能体过滤干扰信息。
3.  **条件性修正指导**：仅在检测到执行偏差（如循环导航、进度停滞）时触发的动态干预机制。通过注入高层指令（如“停止搜索袜子，立即进入结算页面”）校准操作智能体行为，既保证正常执行时的自主性，又能在异常时及时纠偏。

现有研究（如ReSum、WebCoach）虽也采用摘要扩展上下文，但ColorBrowserAgent的创新点在于**赋予摘要智能体双智能体监督角色**。其功能超越被动压缩，通过修正指导实现主动干预，实现了“下一步做什么”（摘要智能体全局规划）与“具体怎么做”（操作智能体精准执行）的职责分离，有效解决单智能体架构的决策漂移问题。

### 4.2 人机知识适配（HITL-KA）
通用大语言模型虽具备强大的通用推理能力，但缺乏适配各类网页独特逻辑的专属先验知识。网页异构性（独特DOM结构、非标交互模式、领域专属流程）形成的“知识鸿沟”，无法仅通过模型推理填补。为避免高昂的站点专属微调成本与繁琐人工标注，本文提出**人机知识适配框架**。该框架摒弃静态标注模式，采用**自主触发机制**，仅在检测到执行异常时请求专家介入，将临时失败转化为可复用的持久化知识，精准投入人力成本。

#### 4.2.1 混合触发机制  
为平衡自主性与人力投入，本文设计混合判别策略，仅在智能体遭遇真实歧义或执行失败时触发专家介入，实现人类专业知识的高效利用：
1.  **规则判别器**：监控执行轨迹中的确定性异常，标记可检测的失效模式，如循环导航（重复访问相同URL序列）、执行停滞（多次操作后状态无变化）、显性DOM错误（如“访问被拒”“商品售罄”提示）。
2.  **视觉语言模型（VLM）判别器**：捕捉更细微的语义失配问题。通过VLM评估当前UI状态与智能体预期动作的一致性，识别规则判别器难以检测的问题，如逻辑矛盾（按钮可见但因表单未填无法点击）、语义无关（搜索结果与查询意图不符）。

#### 4.2.2 知识注入与持久化
触发警报后，系统暂停执行并进入交互模式，知识适配流程分为两个阶段：
1.  **专家介入**：人类专家基于当前截图、可访问性树和近期动作历史，分析失败场景并生成简洁的“知识提示”，将缺失的领域知识转化为可执行的自然语言指令（如“在此结算页面，必须先选择配送方式，‘提交订单’按钮才会激活”）。
2.  **知识库整合**：将知识提示索引并存储至**自适应知识库（AKB）**。后续在相同或语义相似站点执行任务时，智能体基于上下文相似度自动检索相关提示，引导操作策略 $\pi_{\text{op}}$ 生成符合领域规则的有效动作。

通过该机制，智能体可逐步积累站点专属逻辑，将异构性挑战转化为可扩展的知识管理任务，无需模型重训练即可在多样化环境中实现鲁棒性能。

### 4.3 工程优化策略
除核心方法外，本文还实现了多项工程优化，这些策略虽在理论框架中常被忽视，但对智能体的真实场景部署至关重要，显著提升了系统的鲁棒性与执行效率。

__动作空间扩展:__
新增 `take_note()` 和 `calculate()` 等高阶原语，支撑多步推理任务：
- `take_note()`：支持智能体实时记录关键信息，减少token消耗，避免页面跳转时的上下文丢失；
- `calculate()`：针对大语言模型算术运算精度不足的问题，将数值计算任务卸载至确定性计算器工具，解决购物比价、预算汇总等场景中的“幻觉计算”错误，提升计算密集型子任务效率。

__环境异常鲁棒容错:__
针对真实网页环境中的不可预测异常（如搜索功能故障、访问频率限制、控件无响应），设计通用鲁棒策略：
1.  **自主恢复与断点续跑**：构建全面监控机制，检测网络波动、API超时、站点临时崩溃等导致的执行停滞。触发后自动执行会话刷新与上下文恢复，实现无缝断点续跑，保障不稳定环境下的任务连续性。
2.  **URL构造执行捷径**：针对全站搜索等高频操作，单纯依赖UI交互易出错且效率低下。通过自适应知识库将搜索意图直接转化为URL参数（如 `fill(search_box, keyword)` → `goto(/search?search=keyword...)`），大幅减少交互步骤，规避UI渲染问题，实现更高效可靠的信息检索。

## 5 实验
本节对ColorBrowserAgent开展全面评估，依次介绍实验配置（含WebArena基准与环境实现细节）、核心实验结果（验证模型达SOTA性能），并通过领域专项分析与消融实验，量化摘要智能体和自适应知识库两大核心组件的贡献。

### 5.1 实验配置
实验基于**WebArena基准**完成，该基准含812个精心设计的任务，覆盖GitLab、Reddit、购物、购物后台管理、地图五大异构领域，任务复杂度从信息检索延伸至复杂状态变更流程。

ColorBrowserAgent的实现基于**BrowserGym框架**，为保障可复现性并消除网络差异，采用Docker容器本地部署WebArena环境。针对地图类任务，对接开源地图（OpenStreetMap）在线服务，突破本地地图环境的搜索功能限制。所有实验均以**GPT-5**为基础大语言模型，温度系数设为0.0以确保评估阶段决策的确定性。

### 5.2 实验结果
ColorBrowserAgent在WebArena基准上实现**71.2%的总体任务成功率**，刷新当前最优纪录。如表1所示，该模型显著超越现有基线模型（含商用闭源模型与开源研究原型），较此前性能最佳的开源模型Claude Code + GBOX MCP提升3.2个百分点，充分验证了本架构设计的优越性。

**表1 WebArena基准SOTA模型性能对比**
| 开源属性 | 智能体模型 | 成功率 |
| ---- | ---- | ---- |
| 是 | AgentSymbiotic | 52.1% |
| 是 | WebOperator | 54.6% |
| 否 | OpenAI Operator | 58.1% |
| 是 | IBM CUGA | 61.7% |
| 否 | DeepSky Agent | 66.9% |
| 是 | Claude Code + GBOX MCP | 68.0% |
| 是 | ColorBrowserAgent（本文） | **71.2%** |

#### 5.2.1 领域专项性能分析
分领域性能细粒度分析（表2）显示，ColorBrowserAgent在**Reddit（87.4%）**与**购物后台管理（76.4%）**领域表现出极强的鲁棒性。这两类场景的特点是导航层级复杂、表单填写逻辑严格，而本文提出的**自适应知识库（AKB）**可有效引导智能体执行合规的交互序列，成为性能突破的关键。

**表2 WebArena基准分领域成功率**
| 领域 | 任务数量 | 成功率 |
| ---- | ---- | ---- |
| Reddit | 111 | 87.4% |
| 购物后台管理 | 182 | 76.4% |
| 购物 | 187 | 72.9% |
| GitLab | 204 | 65.7% |
| 地图 | 128 | 55.9% |
| 总计 | 812 | 71.2% |

模型在 **地图领域（55.9%）** 的性能相对偏低，源于该领域界面视觉动态性强、DOM信息稀疏，天然存在较高的处理难度。即便如此，ColorBrowserAgent仍较WebOperator基线模型提升16.6个百分点，印证了本文提出的“状态跟踪+知识检索”一体化框架，即便在视觉锚定难度较高的场景下，也能有效维持执行稳定性。

### 5.3 消融实验
为精准量化各核心组件的贡献，基于**WebArena-Lite**子集（含165个任务，与全量基准分布一致）开展消融实验。所有实验变体均采用GPT-5作为基础模型，以排除模型差异对架构评估的干扰，共设置4组实验：
1.  **基线模型（Vanilla Agent）**：仅依赖原始观测历史与GPT-5固有推理能力，不含本文提出的任何模块；
2.  **仅知识增强**：基线模型集成自适应知识库（AKB），支持领域知识检索，但无记忆压缩功能；
3.  **仅摘要增强**：基线模型集成摘要智能体，实现渐进式摘要与纠偏指导，但无外部知识注入；
4.  **完整模型（ColorBrowserAgent）**：同时集成自适应知识库与摘要智能体的全功能架构。

**表3 WebArena-Lite消融实验结果（165个任务）**
| 模型变体 | 摘要智能体 | 自适应知识库 | 成功率 | 性能提升 |
| ---- | ---- | ---- | ---- | ---- |
| 基线模型 | ✗ | ✗ | 61.7% | - |
| 仅知识增强 | ✗ | ✓ | 68.8% | +7.1% |
| 仅摘要增强 | ✓ | ✗ | 65.4% | +3.7% |
| ColorBrowserAgent（完整） | ✓ | ✓ | **72.6%** | **+10.9%** |

实验结果揭示两大核心结论：
1.  **领域知识是性能瓶颈的关键解**：仅知识增强变体实现7.1%的最大单模块性能提升，表明对于通用智能体而言，“掌握如何交互”（站点专属先验知识）的重要性远超“明确做什么”，自适应知识库有效填补了异构环境下的知识鸿沟；
2.  **长程连贯性是复杂任务的核心保障**：仅摘要增强变体实现3.7%的性能提升，定性分析表明，对于步数超过15的长程任务，该模块可有效避免基线模型常见的上下文溢出与幻觉问题，是维持长程执行稳定性的关键。

## 6 实践启示与未来展望
ColorBrowserAgent的研发与部署，为以人为本的自主智能系统设计提供了宝贵经验，也揭示了智能体融入人类工作流的实际挑战。

#### 6.1 面向可信协作的工程优化
创新架构固然引人注目，但实践表明，严谨的工程优化才是用户信任的基石。诸如网络超时鲁棒容错、原子化动作原语、防御性导航策略等基础改进，是将脆弱原型转化为可靠数字协作伙伴的关键。在人机协作场景中，可靠性不仅是一项指标，更是技术落地的前提——只有在真实环境中持续稳定运行的智能体，才能获得用户的任务托付。

#### 6.2 人机共生协作模式
实验结果证实，人在环（HITL）机制绝非兜底方案，而是实现**协同自主**的高效范式。通过智能体主动请求协助，构建起人机共生闭环：AI负责规模化执行标准化任务，人类则针对边缘场景提供高阶直觉判断。未来研究的核心方向是优化交互成本，设计动态识别**最小必要干预**的方法，在最大化利用专家先验知识的同时，最小化人类认知负担，最终实现智能体随用户指导持续进化、逐步降低人工介入需求的目标。

#### 6.3 基础模型作为推理核心
研究进一步印证，强大的基础模型是精准解析人类意图的关键。智能体对新站点的泛化能力，高度依赖底层大语言模型的推理性能。以GPT-5为例，其可有效衔接抽象的人类指令与具体的界面操作。随着基础模型的迭代，未来智能体有望从“执行点击操作”升级为“理解用户偏好的工作流”，实现更个性化、更具直觉性的自动化。

## 7 结论
本文提出ColorBrowserAgent框架，旨在弥合网页自动化领域中自主能力与以人为本可靠性之间的鸿沟。该框架融合**渐进式进度摘要**与**人在环知识适配**两大核心机制，在WebArena基准测试中刷新最优性能纪录，验证了通过协同模式可构建兼具鲁棒性与适应性的智能体系统。

研究挑战了单纯依赖“黑箱自主”的主流思路，提出**协同自主**范式——以强大基础模型为核心，辅以结构化记忆与专家引导先验知识，是实现工业化级实用价值的更可行路径。该设计理念以人类用户为核心，确保智能体始终是透明可控、持续进化的工具。

展望未来，从静态自动化向动态终身学习的跨越，将是下一代智能体的核心突破方向。我们期待未来的智能体不仅能执行预设任务，更能通过交互自主构建对网页环境的认知，从被动工具进化为与人类用户共同成长的自适应数字伙伴。